# TP1 - Import de formats et accès aux API
## Des petits scripts python pour nous aider

In [ ]:
from pathlib import Path
import pandas as pd
import geopandas as gpd

### 0 - Lire un jeu de donnée au format CSV avec geopandas

In [ ]:
dossier_courant = Path.cwd()
dossier_donnees = dossier_courant / "data"
dossier_sortie = dossier_donnees / "output"
dossier_sortie.mkdir(parents=True, exist_ok=True)

In [18]:
print(f"Dossier de travail actuel : {dossier_courant}")

Dossier de travail actuel : /home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1


In [19]:
fichier_csv = dossier_donnees / "human_chamois_encounters.csv"
colonne_wkt = "eca_h"
crs_source = "EPSG:2154"

In [20]:
df = pd.read_csv(fichier_csv)
geometry = gpd.GeoSeries.from_wkt(df[colonne_wkt], crs=crs_source)

if colonne_wkt != "geometry":
    df = df.drop(columns=[colonne_wkt])

gdf = gpd.GeoDataFrame(df, geometry=geometry)

In [7]:
gdf.head()

,id_encounter,type,id_indiv_human,id_indiv_animal,id_traj_human,id_traj_animal,date_human,date_animal,earliest_human,latest_human,earliest_animal,latest_animal,eca_a,geometry
0,1,Potential,19934,1,20533,20,2020-06-27,2013-07-09,06:21:08,06:23:08,06:20:34,06:30:41,"POLYGON((953125.181568785 6508201.833860811,95...","POLYGON ((952977.158 6508367.925, 952897.225 6..."
1,2,Potential,19220,1,20571,20,2020-07-07,2013-07-09,13:52:44,13:53:47,13:50:51,14:00:26,"POLYGON((953051.7089725762 6508157.022283436,9...","POLYGON ((952878.158 6508338.26, 952876.251 65..."
2,3,Potential,21741,1,21741,20,2020-06-23,2013-07-09,13:44:14,13:45:30,13:40:47,13:50:51,"POLYGON((953089.0731599241 6508149.807691646,9...","POLYGON ((952893.81 6508342.013, 952891.886 65..."
3,4,Potential,19376,21,19558,21,2018-08-12,2012-08-05,15:16:23,15:34:35,15:00:48,16:00:47,"POLYGON((948739.8833260989 6514643.379612487,9...","POLYGON ((948499.153 6514567.832, 948498.945 6..."
4,5,Potential,19193,21,19590,21,2012-07-22,2012-08-05,11:28:59,11:41:41,11:00:31,12:00:53,"POLYGON((948692.9600541465 6514687.195881412,9...","POLYGON ((948518.354 6514642.793, 948518.436 6..."


In [8]:
print("Lignes, colonnes :", gdf.shape)
print("CRS :", gdf.crs)
print("Emprise :", gdf.total_bounds)
print("\nTypes de géométries :", gdf.geom_type.value_counts())
print("\nGéométries invalides :", (~gdf.is_valid).sum())

Lignes, colonnes : (63759, 14)
CRS : EPSG:2154
Emprise : [ 947550.02448383 6505444.23466601  955135.96371892 6517307.34002861]

Types de géométries : Polygon         45353
MultiPolygon    18406
Name: count, dtype: int64

Géométries invalides : 0


### 1 - Convertir vers un autre format de données

In [9]:
nom_base = fichier_csv.stem

In [10]:
gdf.to_file(dossier_sortie / f"{nom_base}.geojson", driver="GeoJSON")
gdf.to_file(dossier_sortie / f"{nom_base}.shp", driver="ESRI Shapefile")
gdf.to_file(dossier_sortie / f"{nom_base}.gpkg", driver="GPKG")

gdf.to_parquet(dossier_sortie / f"{nom_base}.parquet")

/tmp/ipykernel_56819/2167896855.py:2: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(dossier_sortie / f"{nom_base}.shp", driver="ESRI Shapefile")
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_encounter' to 'id_encount'
  ogr_write(
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_human' to 'id_indiv_h'
  ogr_write(
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_animal' to 'id_indiv_a'
  ogr_write(
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/

### 2 - Mesures et analyses des fichiers

In [15]:
import time

In [11]:
dossier_sortie_mesures = dossier_sortie / "mesures"
dossier_sortie_mesures.mkdir(parents=True, exist_ok=True)

In [12]:
formats = {
    "Shapefile": (dossier_sortie_mesures / f"{nom_base}.shp", "ESRI Shapefile"),
    "GeoJSON": (dossier_sortie_mesures / f"{nom_base}.geojson", "GeoJSON"),
    "GeoPackage": (dossier_sortie_mesures / f"{nom_base}.gpkg", "GPKG"),
}

In [ ]:
rows = []

for nom, (path, driver) in formats.items():
    t0 = time.perf_counter()
    gdf.to_file(path, driver=driver)
    t_write = time.perf_counter() - t0

    t0 = time.perf_counter()
    _ = gpd.read_file(path)
    t_read = time.perf_counter() - t0

    if nom == "Shapefile":
        size_mb = sum(f.stat().st_size for f in dossier_sortie.glob("zones.*")) / 1e6
    else:
        size_mb = path.stat().st_size / 1e6

    rows.append({
        "Format": nom,
        "Taille (Mo)": round(size_mb, 2),
        "Écriture (s)": round(t_write, 3),
        "Lecture (s)": round(t_read, 3),
    })

parquet_path = dossier_sortie_mesures / f"{nom_base}.parquet"
t0 = time.perf_counter()
gdf.to_parquet(parquet_path, index=False)
t_write = time.perf_counter() - t0

t0 = time.perf_counter()
_ = gpd.read_parquet(parquet_path)
t_read = time.perf_counter() - t0

rows.append({
    "Format": "GeoParquet",
    "Taille (Mo)": round(parquet_path.stat().st_size / 1e6, 2),
    "Écriture (s)": round(t_write, 3),
    "Lecture (s)": round(t_read, 3),
})

/tmp/ipykernel_56819/206250929.py:5: UserWarning: Column names longer than 10 characters will be truncated when saved to ESRI Shapefile.
  gdf.to_file(path, driver=driver)
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_encounter' to 'id_encount'
  ogr_write(
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_human' to 'id_indiv_h'
  ogr_write(
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:733: RuntimeWarning: Normalized/laundered field name: 'id_indiv_animal' to 'id_indiv_a'
  ogr_write(
/home/VPech/Documents/cours-2026-2027/DATA-03-ingenierie-de-la-donnee/tp-data/tp1/.venv/lib/python3.15/site-packages/pyogrio/raw.py:7

In [14]:
print(pd.DataFrame(rows).to_string(index=False))

    Format  Taille (Mo)  Écriture (s)  Lecture (s)
 Shapefile         0.00         1.743        1.000
   GeoJSON       580.05        24.290       14.718
GeoPackage       284.20         1.935        1.206
GeoParquet       147.76         1.376        0.853


### 3 - Récupérer des entitées dans OSM

In [26]:
import osmnx as ox

#### Pour un type de POI 

In [ ]:
fichier_zone = Path("data/parc.gpkg")
couche_zone = None  # "limites_parc" si le GPKG a plusieurs couches
cle_osm = "amenity"
valeur_osm = "drinking_water"
fichier_sortie = Path("data/objets_osm.gpkg")
nom_couche_sortie = f"{cle_osm}_{valeur_osm}"

zone = gpd.read_file(fichier_zone, layer=couche_zone)
polygone_wgs84 = zone.to_crs("EPSG:4326").geometry.union_all()

gdf = ox.features_from_polygon(polygone_wgs84, tags={cle_osm: valeur_osm})
gdf = gdf.to_crs("EPSG:2154")
gdf = gpd.clip(gdf, zone.to_crs("EPSG:2154"))

fichier_sortie.parent.mkdir(parents=True, exist_ok=True)
gdf.to_file(fichier_sortie, layer=nom_couche_sortie, driver="GPKG")

print(f"{len(gdf)} objets exportés vers {fichier_sortie} (couche '{nom_couche_sortie}').")

6 objets exportés vers data/objets_osm.gpkg (couche 'amenity_drinking_water').


#### Pour plusieurs type de POI

In [29]:
tags_poi = {
    "amenity": ["drinking_water", "parking", "picnic_table"],
    "natural": ["spring", "peak", "saddle"],
    "tourism": ["viewpoint", "picnic_site", "alpine_hut", "wilderness_hut"],
}

In [30]:
poi = ox.features_from_polygon(polygone_wgs84, tags=tags_poi).to_crs("EPSG:2154")

poi_parc = gpd.clip(poi, zone)

cols = [c for c in ["name", "amenity", "natural", "tourism", "geometry"] if c in poi_parc.columns]
poi_clean = poi_parc[cols].copy()
poi_clean["geometry"] = poi_clean.geometry.centroid
print(f"Nombre d'objets récupérés dans le parc : {len(poi_clean)}")
poi_clean.to_file("poi_parc.gpkg", layer="poi_eau_photo_piquenique", driver="GPKG")

Nombre d'objets récupérés dans le parc : 60
